# 03 — Query-Only Tier-Mapped Pareto Router & OOD Evaluation (`RouterBench`)

This notebook trains and evaluates a query-only PCA + LightGBM router on `data/processed/routerbench_features.parquet`, sweeps cost-weight $\lambda \in [0.00, 1.00]$, and compares in-domain (`test`) and out-of-distribution (`ood_test`, held-out `mbpp`) cost-quality Pareto curves against `Always Frontier`, `Always Small`, and `Random Router` baselines.

## Routing Policy
- **Tier selection rule:** route to `tier1` (small model) when $\hat{p}(\text{tier1\_sufficient}=1 \mid x_{\text{query}}) \ge 1 - \lambda$, else `tier2` (frontier model).
- **Metrics:** trapezoidal `AUC-QC` over normalized cost $\in [0, 1]$ vs. average quality, and maximum cost savings retaining $\ge 95\%$ of `Always Frontier` quality.

In [ ]:
import json
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML, display

ROOT = (Path("..") if Path("../src").exists() else Path(".")).resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.router.pareto import run_router_pipeline
from src.router.trainer import predict_tier1_probability, select_tier

FEATURES_PARQUET = ROOT / "data/processed/routerbench_features.parquet"
CONFIG_YAML = ROOT / "config/routerbench.yaml"
ARTIFACTS_DIR = ROOT / "artifacts"

ROUTER_PKL = ARTIFACTS_DIR / "router.pkl"
PARETO_JSON = ARTIFACTS_DIR / "pareto_sweep.json"

if not (ROUTER_PKL.exists() and PARETO_JSON.exists()):
    run_router_pipeline(FEATURES_PARQUET, CONFIG_YAML, ARTIFACTS_DIR)

sweep = json.loads(PARETO_JSON.read_text(encoding="utf-8"))
with ROUTER_PKL.open("rb") as fh:
    bundle = pickle.load(fh)

print(
    f"Status: {sweep['status']} | Splits: train={sweep['n_train']}, "
    f"test={sweep['n_test']}, ood={sweep['n_ood']} ({sweep['ood_tasks']})"
)
savings_95 = sweep["cost_savings_at_95pct_quality"]
savings_95_str = "N/A" if savings_95 is None else f"{savings_95:.4f}"
print(
    f"In-domain AUC-QC={sweep['auc_qc']:.4f} | "
    f"Cost savings @95% quality={savings_95_str}"
)

## 1. Baseline Comparison (In-Domain `test` & OOD `mbpp`)

In [ ]:
def build_baseline_table(baselines: dict, split_label: str) -> pd.DataFrame:
    rows = []
    for name in ("always_frontier", "always_small", "random_router", "learned_pareto_router"):
        b = baselines[name]
        rows.append(
            {
                "Split": split_label,
                "Method": name,
                "Avg Quality": round(b["avg_quality"], 4),
                "Avg Cost ($)": round(b["avg_cost"], 6),
                "Normalized Cost": round(b["normalized_cost"], 4),
                "Cost Savings": round(b["cost_savings"], 4),
                "Tier-1 Rate": round(b["tier1_rate"], 4),
            }
        )
    return pd.DataFrame(rows)

display(build_baseline_table(sweep["baselines"], "test (in-domain)"))
if sweep["ood_available"]:
    display(build_baseline_table(sweep["ood_baselines"], "ood_test (mbpp)"))

## 2. In-Domain & OOD Pareto Frontier Plots

In [ ]:
def render_pareto_svg(curve: list[dict], baselines: dict, title: str, stroke: str) -> str:
    w, h, pad = 560, 270, 44
    pw, ph = w - 2 * pad, h - 2 * pad
    df_c = pd.DataFrame(curve).sort_values("normalized_cost")

    def x_pos(norm_cost: float) -> float:
        return pad + max(0.0, min(1.0, norm_cost)) * pw

    def y_pos(qual: float) -> float:
        return h - pad - max(0.0, min(1.0, qual)) * ph

    pts = " ".join(
        f"{x_pos(float(r['normalized_cost'])):.1f},{y_pos(float(r['avg_quality'])):.1f}"
        for _, r in df_c.iterrows()
    )
    q95 = 0.95 * baselines["always_frontier"]["avg_quality"]
    y95 = y_pos(q95)
    rnd = baselines["random_router"]
    lrn = baselines["learned_pareto_router"]
    return (
        f'<svg width="{w}" height="{h}" style="border:1px solid #e2e8f0;background:#fff;margin-bottom:10px;">'
        f'<line x1="{pad}" y1="{y95:.1f}" x2="{w - pad}" y2="{y95:.1f}" '
        f'stroke="#94a3b8" stroke-dasharray="4,4" stroke-width="1.5"/>'
        f'<polyline fill="none" stroke="{stroke}" stroke-width="2.5" points="{pts}"/>'
        f'<circle cx="{x_pos(rnd["normalized_cost"]):.1f}" cy="{y_pos(rnd["avg_quality"]):.1f}" '
        f'r="4.5" fill="#f97316"/>'
        f'<circle cx="{x_pos(lrn["normalized_cost"]):.1f}" cy="{y_pos(lrn["avg_quality"]):.1f}" '
        f'r="5" fill="#111827"/>'
        f'<text x="{pad}" y="20" font-size="12" font-family="monospace">{title}</text>'
        f'<text x="{pad}" y="{h - 12}" font-size="11" font-family="monospace">'
        f'x=Normalized Cost [0..1] | y=Avg Quality [0..1] | dashed=95% Frontier | orange=Random | black=lambda={sweep["default_lambda"]}</text>'
        f'</svg>'
    )

html_parts = [
    render_pareto_svg(
        sweep["curve"],
        sweep["baselines"],
        f"In-Domain Test Pareto (AUC-QC={sweep['auc_qc']:.3f})",
        "#0f766e",
    )
]
if sweep["ood_available"]:
    html_parts.append(
        render_pareto_svg(
            sweep["ood_curve"],
            sweep["ood_baselines"],
            f"OOD mbpp Pareto (AUC-QC={sweep['ood_auc_qc']:.3f})",
            "#0284c7",
        )
    )
display(HTML("<br/>".join(html_parts)))
display(pd.DataFrame(sweep["curve"]).round(4))

## 3. Sample Query Routing Inference

In [ ]:
df_test = pd.read_parquet(FEATURES_PARQUET)
df_test = df_test[df_test["split"] == "test"].reset_index(drop=True)
sample_meta = df_test.iloc[[0]][bundle.query_meta_cols]
sample_emb = np.asarray(df_test.iloc[0]["query_embedding"], dtype=np.float32)
prob = float(predict_tier1_probability(bundle, sample_emb, sample_meta)[0])
tier = select_tier(prob, lambda_cost=bundle.default_lambda)
print(f"Sample test[0] -> p(tier1_sufficient)={prob:.4f}, lambda={bundle.default_lambda}, tier={tier}")